# 🤖 Diario LowCode — AetherNet

**Autor:** Andres Felipe Martinez Henao

**Proyecto:** AetherNet IoT & Autonomous Rover — Diario de automatización LowCode, notificaciones HU-02 y deuda Node-RED (Sprint 1-4).

**Links clave:**
- `automation/flows/intrusion_alert.json` — flujo Node-RED referencia (deuda 2026-09-09)
- `docs/Automatizacion-LowCode/backlog-lowcode.md` — backlog LOW-01..LOW-07 (Tuya cancelado ADR-001)
- `docs/Automatizacion-LowCode/roadmap-lowcode.md` — roadmap Telegram + Node-RED deuda
- `docs/prd.md` HU-02 — intrusión láser → notificación <2 s + LED rojo 3 s
- `backend/app/routers/events.py` — `POST /api/security-events` RF-4.1 Telegram directo
- `firmware/mega-access/src/laser.cpp` — `SECURITY:{"event_type":"intrusion"}` + LED 44/45/46

> Trazabilidad: HU-02 / RF-4.1 (Telegram) / RF-4.2 Won't (Tuya ADR-001) / LOW-02 deuda Node-RED.
- `docs/prd.md` HU-02 — prd.md HU-02 trazabilidad


## Estado 2026-09-09 — LOW-02 deuda

**LOW-02 Node-RED DEUDA TÉCNICA (2026-09-09):** flujo `automation/flows/intrusion_alert.json` queda solo como **referencia JSON exportable** — no hay runtime Node-RED desplegado esta iteración (asesor). HU-02 vigente: **LED RGB rojo 3 s** (`laser.cpp`) + **Telegram Bot API directo** (`POST /api/security-events`) + **App dashboard**.
MEGA laser → UART SECURITY: → Gateway → MQTT aethernet/seguridad/intrusion → Telegram + LED rojo + App dashboard


**Flujo HU-02 vigente (sin Node-RED):**

```mermaid
flowchart LR
  MEGA[MEGA 2560<br/>Laser KY-008 TX8 → LDR 7<br/>laser.cpp handleLaser 50ms]
  MEGA -->|UART 38400<br/>SECURITY:{"event_type":"intrusion"}| GW[Gateway ESP32<br/>UART2 16/17]
  GW -->|MQTT publish<br/>aethernet/seguridad/intrusion| MQTT[(Mosquitto<br/>1883 / 9001<br/>aethernet/#)]
  MQTT -->|suscripción| TG[Telegram Bot API<br/>sendMessage parse_mode Markdown<br/>RF-4.1 HU-02]
  MQTT -->|suscripción| LED[LED RGB 44/45/46<br/>rojo 3s laser.cpp]
  MQTT -->|suscripción| APP[App AetherControl<br/>dashboard + AccessEvent]
  MQTT -.->|deuda Sprint 4| NR[Node-RED<br/>intrusion_alert.json<br/>mqtt in → function → http-telegram]
```

> Node-RED en línea punteada = deuda técnica. Ver `docs/Automatizacion-LowCode/backlog-lowcode.md` LOW-02/LOW-03/LOW-05 y `docs/Automatizacion-LowCode/README.md`.


In [ ]:
import json, pathlib
flow = json.load(open("automation/flows/intrusion_alert.json"))
print(json.dumps(flow, indent=2)[:1200])
print("--- backend telegram ---")
txt = pathlib.Path("backend/app/routers/events.py").read_text()
for i,l in enumerate(txt.splitlines(),1):
    if "telegram" in l.lower() or "security" in l.lower():
        print(f"{i}: {l}")

## Payloads

**MQTT `aethernet/seguridad/intrusion` (Gateway → suscriptores):**

| Campo | Ejemplo | Origen |
|---|---|---|
| `event_type` | `"intrusion"` | `laser.cpp:32 SECURITY:` |
| `sensor` | `"laser-01"` | `function-parse-intrusion` |
| `location` | `"entrance"` | payload Node-RED |
| `timestamp` | `"2026-09-09T14:28:00Z"` | `new Date().toISOString()` |
| `severity` | `"high"` | `function-parse-intrusion` |

```json
{
  "event_type": "intrusion",
  "sensor": "laser-01",
  "location": "entrance",
  "timestamp": "2026-09-09T14:28:00Z",
  "severity": "high"
}
```

**UART MEGA → Gateway `SECURITY:` (fuente verdad firmware):**

```text
SECURITY:{"event_type":"intrusion","sensor":"laser-01"}
```

- MEGA `laser.cpp:32` `handleLaser()` 50 ms CHECK / 3 s COOLDOWN → `uart_protocol.cpp:40` `SECURITY:` → Gateway `gateway-esp32.ino:366 handleSecurityEvent` → `POST /api/security-events 201` + `MQTT aethernet/seguridad/intrusion`.
- Node-RED deuda: `mqtt in aethernet/seguridad/intrusion qos 1 json` → `function-parse-intrusion` parsea `sensor_id/location/timestamp` → `telegram-alert` formatea Markdown → `http-telegram POST https://api.telegram.org/bot$globalContext("telegram_bot_token")/sendMessage`.
- Telegram directo vigente: `backend/app/routers/events.py:141 create_security_event` `POST /api/security-events` con `SecurityEventCreate` (`event_type`, `severity`, `description`) — visto en celda 2 `grep telegram|security`.


## 📸 Capturas

> Checklist vivo: `docs/Automatizacion-LowCode/README.md` — si falta captura deja `![CAPTURA PENDIENTE]`, no borres la línea.

![CAPTURA PENDIENTE](../../docs/Automatizacion-LowCode/capturas/botfather.png) — BotFather /newbot
![CAPTURA PENDIENTE](../../docs/Automatizacion-LowCode/capturas/telegram-alert.png) — mensaje intrusión real aethernet/seguridad/intrusion
![CAPTURA PENDIENTE](../../docs/Automatizacion-LowCode/capturas/node-red-flow.png) — flujo intrusion_alert.json importado (deuda Sprint 4)
![FOTO PENDIENTE](../../docs/Automatizacion-LowCode/capturas/led-rojo-intrusion.jpg) — LED 44/45/46 rojo 3s laser.cpp

- **Cómo tomar:** abre Telegram, crea bot con `@BotFather` → `/newbot` → copia `token` (guardar en `process.env.TELEGRAM_TOKEN`, jamás en JSON) → anota `chat_id` → publica intrusión simulada `mosquitto_pub -h 192.168.1.14 -t aethernet/seguridad/intrusion -m '{"sensor_id":"laser-01","location":"entrance"}'` → screenshot del chat con `🚨 ALERTA DE INTRUSIÓN 🚨` + hora/sensor/severidad.
- **Guardar en:** `docs/Automatizacion-LowCode/capturas/` — nombres exactos `botfather.png` `telegram-alert.png` `node-red-flow.png` `led-rojo-intrusion.jpg`.
- **Validador:** celda 5 lista `*.png/*.jpg` + tamaño; si vacío imprime `CAPTURAS PENDIENTES — ver docs/Automatizacion-LowCode/README.md`.


In [ ]:
from pathlib import Path
for p in sorted(Path("docs/Automatizacion-LowCode/capturas").glob("*")):
    print(p.name, p.stat().st_size)
if not list(Path("docs/Automatizacion-LowCode/capturas").glob("*.png")):
    print("CAPTURAS PENDIENTES — ver docs/Automatizacion-LowCode/README.md")

## Siguientes pasos

| Sprint | Tarea | Estado | Depende de |
|---|---|---|---|
| **4** | Bot Telegram directo `LOW-03` `sendMessage` + `parse_mode Markdown` vía `POST /api/security-events` | 🔜 | `backend/app/routers/events.py:141` |
| **4** | E2E intrusión `LOW-05` laser → UART SECURITY: → MQTT aethernet/seguridad/intrusion → Telegram <2 s + LED rojo 3 s | 🔜 | LOW-03 + `laser.cpp` |
| **4** | Node-RED `LOW-02` importar `intrusion_alert.json` + broker `aethernet/seguridad/intrusion` + `process.env.TELEGRAM_TOKEN` — solo si retoma deuda | 📋 deuda 2026-09-09 | `automation/flows/intrusion_alert.json` |
| **Siempre** | Tomar capturas `botfather.png` `telegram-alert.png` `node-red-flow.png` `led-rojo-intrusion.jpg` → marcar `[x]` en `docs/Automatizacion-LowCode/README.md` | 📸 pendiente | este notebook celda 4/5 |

**Refs:**
- `docs/Automatizacion-LowCode/backlog-lowcode.md` — LOW-01..LOW-07 (Tuya ADR-001 cancelado)
- `docs/Automatizacion-LowCode/roadmap-lowcode.md` — Tabla Telegram/Node-RED/Tuya/FastAPI/LED + Estado 2026-09-09
- `docs/Automatizacion-LowCode/README.md` — checklist capturas/logs LowCode
- `automation/flows/intrusion_alert.json` — `mqtt in aethernet/seguridad/intrusion` → `function-parse-intrusion` → `http-telegram`
- `docs/prd.md` HU-02 + `docs/requirements.md` RF-4.1/RF-4.2 + `docs/adr/adr-001-cancelacion-tuya.md`
- `docs/sprints.md` Sprint 4 — si retoma Node-RED, desplegar runtime y probar `mosquitto_pub` intrusión simulada

> Ejecuta celdas 2 y 5 para verificar flujo referencia + capturas antes de sprint review. HU-02 sin Node-RED: `laser.cpp` + `POST /api/security-events` + `aethernet/seguridad/intrusion`.

Sprint 4 si retoma Node-RED — ver `docs/Automatizacion-LowCode/README.md`
